# ReSnap — Retrain v4 di Kaggle (YOLOv8s) + evaluasi test final

Versi Kaggle dari `ReSnap_Retrain_V4.ipynb`. Dataset v4 (dataset v3 + foto Open Images yang sudah disaring) sudah dibuat di Colab, jadi notebook ini langsung training.

**Persiapan (sekali):**
1. Buat Kaggle Dataset baru (Datasets > New Dataset), isi dengan 3 file dari Drive:
   - `dataset_v4.zip`
   - `ReSnap_Kandidat_Tambah_Foto_Udang.zip` (di `review_anotasi/tambahan_foto_udang_.../`)
   - `resnap.zip` (folder repo, termasuk `models/best.pt` v3)

   Kaggle boleh mengekstrak ZIP-nya otomatis; notebook ini bisa membaca keduanya.
2. Di notebook: **Add Input** > pilih dataset tadi. Settings: **Accelerator = GPU T4 x2 atau P100**, **Internet = On** (butuh akun terverifikasi nomor HP).
3. Jalankan lewat **Save Version > Save & Run All (Commit)** supaya tetap jalan walau browser ditutup (batas 12 jam; perkiraan total 4–6 jam).

**Isi run:** training YOLOv8s, diagnosa overfit/underfit, perbandingan v3 vs v4 di val dan holdout Open Images, keputusan terima/tolak berdasar 4 syarat, lalu evaluasi **test set satu kali** untuk model terpilih (v4 bila diterima, selain itu v3), ablasi preprocessing, dan export ONNX. Semua hasil dikemas ke `hasil_v4.zip` di tab Output.

## 1. Paket, input, dan konfigurasi

In [ ]:
%pip -q install ultralytics==8.3.228

In [ ]:
from pathlib import Path
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working')
TMP = Path('/tmp/resnap')          # data besar di sini supaya tidak ikut Output
DEVICE = '0'

In [ ]:
import json, random, shutil, sys, zipfile
from collections import Counter

import torch
assert torch.cuda.is_available(), 'GPU tidak aktif. Settings > Accelerator > GPU, lalu jalankan ulang.'
print('GPU:', torch.cuda.get_device_name(0))

def find_first(pred, what):
    hits = sorted((p for p in INPUT.rglob('*') if pred(p)), key=lambda p: len(p.parts))
    if not hits:
        raise FileNotFoundError(f'{what} tidak ditemukan di {INPUT}. Cek Add Input.')
    return hits[0]

def materialize(zip_name, is_root, dest, what):
    # Pakai folder hasil ekstrak Kaggle bila ada, kalau tidak ekstrak ZIP-nya ke dest.
    if dest.exists() and is_root(dest):
        return dest
    try:
        src = find_first(lambda p: p.is_dir() and is_root(p), what)
        shutil.copytree(src, dest, dirs_exist_ok=True)
    except FileNotFoundError:
        z = find_first(lambda p: p.name == zip_name, what)
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)
        inner = [p for p in dest.rglob('*') if p.is_dir() and is_root(p)]
        if not is_root(dest) and inner:  # ZIP berisi satu folder induk
            return sorted(inner, key=lambda p: len(p.parts))[0]
    assert is_root(dest), f'Struktur {what} tidak dikenali'
    return dest

TMP.mkdir(parents=True, exist_ok=True)
REPO = materialize('resnap.zip', lambda p: (p / 'scripts/data_v4.py').is_file(), TMP / 'repo', 'Repo resnap')
base = materialize('ReSnap_Kandidat_Tambah_Foto_Udang.zip',
                   lambda p: (p / 'test/images').is_dir() and (p / 'reports/class_catalog.json').is_file(),
                   TMP / 'base', 'Dataset kandidat v3')
data = materialize('dataset_v4.zip', lambda p: (p / 'holdout_oi/images').is_dir(), TMP / 'data', 'Dataset v4')

sys.path.insert(0, str(REPO / 'scripts'))
import data_v4 as dv
from diagnose_fit import analyze_curves, load_results, plot as plot_curves, verdict
from evaluate_export import evaluate
from ultralytics import YOLO

V3_WEIGHTS = REPO / 'models/best.pt'
V3_SHA = 'ac24d434cc97d12f3fd3ed204d2fefda8cccfc322422a8c55672190a0615da98'
assert dv.sha256_file(V3_WEIGHTS) == V3_SHA, 'models/best.pt di repo bukan v3'

EXPECTED = {'base': {'train': 4500, 'val': 1069, 'test': 654},
            'data': {'train': 5318, 'val': 1069, 'holdout_oi': 78}}
found = {'base': {s: len(list((base / s / 'images').iterdir())) for s in EXPECTED['base']},
         'data': {s: len(list((data / s / 'images').iterdir())) for s in EXPECTED['data']}}
print(found)
assert found == EXPECTED, 'Jumlah foto tidak cocok dengan dataset yang dibuat di Colab'

RUN_PROJECT = OUTPUT / 'training'
RUN_NAME = 'resnap_yolov8s_30kelas_v4_oi'
RUN_DIR = RUN_PROJECT / RUN_NAME
OUT = OUTPUT / 'evaluasi_v4'
OUT.mkdir(parents=True, exist_ok=True)
WORK = TMP
OVERSAMPLE_COPIES = 1
OI_TRAIN_REQUEST = {'Potato': 700, 'Shrimp': 700, 'Mushroom': 350}

yaml_train = dv.write_yaml(data / 'data.yaml', data, 'train/images', 'val/images')
yaml_holdout = dv.write_yaml(data / 'holdout.yaml', data, 'holdout_oi/images', 'holdout_oi/images')
stats = {'train_v3': dv.count_split(base / 'train'), 'train_v4': dv.count_split(data / 'train'),
         'val': dv.count_split(data / 'val'), 'holdout_oi': dv.count_split(data / 'holdout_oi')}
dv.save_json(OUT / 'jumlah_data.json', stats)
print('Siap. Kentang train:', stats['train_v4']['per_class']['potato'],
      '| udang train:', stats['train_v4']['per_class']['shrimp'])

## 2. Training YOLOv8s

Setelan sama dengan versi Colab: AdamW lr 0.0003 (nilai yang sebenarnya dipakai v3), augmentasi sama dengan v3, maksimal 100 epoch, early stopping patience 20, `best.pt` dipilih dari val. Kaggle tidak bisa melanjutkan run dari sesi lain, jadi `RESUME` dimatikan.

In [ ]:
RESUME = False   # di Kaggle, run selalu dimulai baru
MAX_EPOCHS = 100
RECIPE = dict(
    epochs=MAX_EPOCHS, patience=20, batch=16, imgsz=640, save=True, save_period=5, cache=False,
    workers=2, pretrained=True, optimizer='AdamW', lr0=0.0003, lrf=0.01, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=3.0, seed=42, deterministic=True, cos_lr=False,
    close_mosaic=10, amp=True, plots=True, val=True,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.35, degrees=10.0, translate=0.1, scale=0.4, shear=0.0,
    perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=0.5, mixup=0.0, copy_paste=0.0,
)
if RESUME:
    YOLO(str(RUN_DIR / 'weights/last.pt')).train(resume=True, device=DEVICE)
else:
    assert not RUN_DIR.exists(), f'{RUN_DIR} sudah ada. Ganti RUN_NAME atau set RESUME=True.'
    RUN_DIR.mkdir(parents=True)
    dv.save_json(RUN_DIR / 'experiment.json', {'base_model': 'yolov8s.pt', 'settings': RECIPE,
                                               'oversample_copies': OVERSAMPLE_COPIES,
                                               'oi_train_request': OI_TRAIN_REQUEST,
                                               'train_images': stats['train_v4']['images']})
    YOLO('yolov8s.pt').train(data=str(yaml_train), project=str(RUN_PROJECT), name=RUN_NAME,
                             exist_ok=True, device=DEVICE, **RECIPE)
V4_WEIGHTS = RUN_DIR / 'weights/best.pt'
dv.save_json(RUN_DIR / 'training_complete.json', {'weights_sha256': dv.sha256_file(V4_WEIGHTS)})
print('Selesai:', V4_WEIGHTS)

## 3. Diagnosa overfit / underfit

In [ ]:
V4_WEIGHTS = RUN_DIR / 'weights/best.pt'
rng = random.Random(42)

def subset_yaml(src_dir, n, name):
    imgs = [p for p in (src_dir / 'images').iterdir() if '_dup' not in p.stem]
    pick = rng.sample(imgs, min(n, len(imgs)))
    dst = WORK / name
    if dst.exists():
        shutil.rmtree(dst)
    (dst / 'images').mkdir(parents=True); (dst / 'labels').mkdir()
    for p in pick:
        shutil.copy2(p, dst / 'images' / p.name)
        shutil.copy2(src_dir / 'labels' / f'{p.stem}.txt', dst / 'labels' / f'{p.stem}.txt')
    return dv.write_yaml(dst / 'data.yaml', dst, 'images', 'images')

yaml_tr_v4 = subset_yaml(data / 'train', 600, 'train_subset_v4')
yaml_tr_v3 = subset_yaml(base / 'train', 600, 'train_subset_v3')

E = {}
E['v4_val'] = evaluate(str(V4_WEIGHTS), yaml_train, 'val', OUT, 'v4_val', DEVICE)
E['v4_train'] = evaluate(str(V4_WEIGHTS), yaml_tr_v4, 'val', OUT, 'v4_train_subset', DEVICE)
E['v3_val'] = evaluate(str(V3_WEIGHTS), yaml_train, 'val', OUT, 'v3_val', DEVICE)
E['v3_train'] = evaluate(str(V3_WEIGHTS), yaml_tr_v3, 'val', OUT, 'v3_train_subset', DEVICE)
E['v4_holdout'] = evaluate(str(V4_WEIGHTS), yaml_holdout, 'val', OUT, 'v4_holdout', DEVICE)
E['v3_holdout'] = evaluate(str(V3_WEIGHTS), yaml_holdout, 'val', OUT, 'v3_holdout', DEVICE)

ref_gap = E['v3_train']['mAP50_95'] - E['v3_val']['mAP50_95']
rows = load_results(RUN_DIR / 'results.csv')
curves = analyze_curves(rows, MAX_EPOCHS)
fit = verdict(curves, E['v4_train']['mAP50_95'], E['v4_val']['mAP50_95'], ref_gap)
plot_curves(rows, OUT / 'kurva_v4.png', curves['best_epoch'], 'Kurva training v4 (YOLOv8s)')
dv.save_json(OUT / 'diagnosa_fit.json', {'kurva': curves, 'gap_v3': ref_gap, **fit, 'evaluasi': E})
print(json.dumps({'kurva': curves, 'gap_v3': round(ref_gap, 4), **fit}, indent=2, ensure_ascii=False))
from IPython.display import Image, display
display(Image(str(OUT / 'kurva_v4.png'), width=900))

## 4. Bandingkan v3 dan v4

In [ ]:
import pandas as pd

def per_class(name):
    return pd.read_csv(OUT / name / 'per_class.csv').set_index('class_name')

overall = pd.DataFrame({k: E[k] for k in ('v3_val', 'v4_val')}).loc[
    ['precision', 'recall', 'f1', 'mAP50', 'mAP75', 'mAP50_95']]
overall['delta_pp'] = 100 * (overall['v4_val'] - overall['v3_val'])
display(overall.style.format(precision=3))

pv3, pv4 = per_class('v3_val'), per_class('v4_val')
cmp = pd.DataFrame({'instances': pv4['instances'], 'AP50_v3': pv3['AP50'], 'AP50_v4': pv4['AP50']})
cmp['delta_pp'] = 100 * (cmp['AP50_v4'] - cmp['AP50_v3'])
display(cmp.sort_values('delta_pp').style.format(precision=3).set_caption('AP50 per kelas di val'))

hv3, hv4 = per_class('v3_holdout'), per_class('v4_holdout')
focus = [c for c in ('potato', 'shrimp', 'mushroom') if c in hv4.index]
hold = pd.DataFrame({'instances': hv4.loc[focus, 'instances'],
                     'AP50_v3': hv3.reindex(focus)['AP50'].fillna(0), 'AP50_v4': hv4.loc[focus, 'AP50'],
                     'recall_v3': hv3.reindex(focus)['recall'].fillna(0), 'recall_v4': hv4.loc[focus, 'recall']})
hold['delta_AP50_pp'] = 100 * (hold['AP50_v4'] - hold['AP50_v3'])
display(hold.style.format(precision=3).set_caption('Holdout Open Images (tidak ikut training)'))
cmp.to_csv(OUT / 'banding_per_kelas_val.csv'); hold.to_csv(OUT / 'banding_holdout.csv')

## 5. Keputusan

v4 diterima bila **semua** syarat terpenuhi:
1. Vonis diagnosa `seimbang` (tidak overfit dan sudah konvergen).
2. mAP50-95 val tidak turun lebih dari 0.5 poin dari v3.
3. AP50 holdout naik minimal 5 poin untuk kentang **dan** udang.
4. Tidak ada kelas val dengan minimal 20 instance yang AP50-nya turun lebih dari 10 poin.

Kalau gagal, petunjuk perbaikannya dicetak di bawah.

In [ ]:
checks = {
    'fit_seimbang': fit['vonis'] == 'seimbang',
    'val_map_tidak_turun': overall.loc['mAP50_95', 'delta_pp'] >= -0.5,
    'kentang_naik': 'potato' in hold.index and hold.loc['potato', 'delta_AP50_pp'] >= 5,
    'udang_naik': 'shrimp' in hold.index and hold.loc['shrimp', 'delta_AP50_pp'] >= 5,
    'tidak_ada_kelas_anjlok': not ((cmp['instances'] >= 20) & (cmp['delta_pp'] < -10)).any(),
}
ACCEPT = all(checks.values())
for k, v in checks.items():
    print(('LULUS ' if v else 'GAGAL ') + k)
tips = {
    'fit_seimbang': {
        'underfit': 'Belum konvergen: training ulang dengan MAX_EPOCHS lebih besar (mis. 150) dan RUN_NAME baru. Resume tidak bisa menambah epoch.',
        'overfit': 'Overfit: OVERSAMPLE_COPIES=0 dan mosaic=1.0, lalu training ulang dengan RUN_NAME baru. best.pt saat ini tetap dari epoch val terbaik.',
    }.get(fit['vonis'], 'Overfit sekaligus belum konvergen: kurangi oversampling (OVERSAMPLE_COPIES=0) dan naikkan MAX_EPOCHS, RUN_NAME baru.'),
    'val_map_tidak_turun': 'Data Open Images menggeser distribusi. Kurangi OI_TRAIN_REQUEST Mushroom atau cek contoh_openimages.png.',
    'kentang_naik': 'Tambah OI_TRAIN_REQUEST Potato (mis. 1200) dan periksa log_openimages_train.csv.',
    'udang_naik': 'Tambah OI_TRAIN_REQUEST Shrimp (mis. 1200) dan periksa log_openimages_train.csv.',
    'tidak_ada_kelas_anjlok': 'Lihat tabel per kelas; kelas yang anjlok mungkin tertutup label Open Images yang hilang.',
}
for k, v in checks.items():
    if not v:
        print('->', tips[k])
dv.save_json(OUT / 'keputusan.json', {'accept': ACCEPT, 'checks': {k: bool(v) for k, v in checks.items()}})
print('\nv4 DITERIMA' if ACCEPT else '\nv4 BELUM DITERIMA, v3 tetap dipakai')

## 6. Evaluasi test final (sekali) untuk model terpilih

Model terpilih = v4 bila diterima, selain itu v3. Test hanya dijalankan di sini, setelah keputusan dibuat, sehingga angkanya tidak ikut memengaruhi pemilihan model. Termasuk ablasi foto mentah vs denoise + CLAHE dan export ONNX.

In [ ]:
import subprocess
CHOSEN = 'v4' if ACCEPT else 'v3'
chosen_weights = V4_WEIGHTS if ACCEPT else V3_WEIGHTS
final_dir = OUTPUT / f'evaluasi_final_{CHOSEN}'
weights_copy = OUTPUT / f'best_{CHOSEN}.pt'
shutil.copy2(chosen_weights, weights_copy)

test_yaml = dv.write_yaml(TMP / 'test.yaml', base, 'train/images', 'val/images', {'test': 'test/images'})
for split in ('test', 'val'):
    cmd = [sys.executable, str(REPO / 'scripts/evaluate_export.py'), '--weights', str(weights_copy),
           '--data', str(test_yaml), '--split', split, '--out', str(final_dir / split),
           '--device', DEVICE, '--ablation'] + (['--onnx'] if split == 'test' else [])
    subprocess.run(cmd, check=True, cwd=REPO)

final = {s: json.loads((final_dir / s / 'ringkasan.json').read_text()) for s in ('test', 'val')}
cols = ['images', 'precision', 'recall', 'f1', 'mAP50', 'mAP75', 'mAP50_95']
display(pd.DataFrame({'test (mentah)': final['test']['raw'], 'test (denoise+CLAHE)': final['test']['enhanced'],
                      'val (mentah)': final['val']['raw'], 'val (denoise+CLAHE)': final['val']['enhanced']})
        .loc[cols].style.format(precision=3).set_caption(f'Model {CHOSEN}'))
print('Selisih ablasi test (poin):', final['test']['delta_pp'])
print('Selisih ablasi val  (poin):', final['val']['delta_pp'])
per_cls = pd.read_csv(final_dir / 'test/test_raw/per_class.csv').sort_values('AP50')
display(per_cls.head(8).style.format(precision=3).set_caption('8 kelas terlemah di test'))
from IPython.display import Image
for f in ('confusion_matrix_normalized.png', 'BoxPR_curve.png'):
    display(Image(str(final_dir / 'test/test_raw' / f), width=700))

## 7. Kemas hasil

Unduh `hasil_v4.zip` dari tab **Output** (kanan), lalu kirim ke Drive. Checkpoint per-epoch tidak ikut supaya ukurannya kecil.

In [ ]:
pack = OUTPUT / 'hasil_v4.zip'
with zipfile.ZipFile(pack, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file() and p != pack and not p.name.startswith('epoch') and p.suffix != '.ipynb':
            z.write(p, p.relative_to(OUTPUT))
print(pack, round(pack.stat().st_size / 1e6, 1), 'MB')
for p in (RUN_DIR / 'weights').glob('epoch*.pt'):
    p.unlink()   # hemat ruang Output
print('Model terpilih:', CHOSEN, '| keputusan v4:', 'DITERIMA' if ACCEPT else 'DITOLAK')